# 00 - Preparation : folds, verification, table d'augmentation

A lancer une fois avant les notebooks de modeles. Construit les folds depuis le CSV de split, fige les listes de train partagees par tous les modeles, verifie la plomberie et exporte la table d'augmentation.

**Le fold 5 est le test : il n'est jamais utilise par la comparaison.**

In [ ]:
# --- Installation (package partage uniquement) ---
!pip install -q pycocotools openpyxl wandb

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
import os, sys
REPO_DIR = "/content/aphids_detection"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/EmmaDub/aphids_detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__)

In [ ]:
# --- Configuration ---
# Les chemins par defaut sont ceux de aphids_det/config.py. Pour les changer,
# decommenter et adapter, puis relancer cfg.refresh().
from pathlib import Path
import aphids_det.config as cfg

# cfg.BASE_DIR  = Path("/content/drive/MyDrive/.../tuile_viz02_640_128")
# cfg.OUT_DIR   = Path("/content/drive/MyDrive/.../puceron_model_article/data")
# cfg.EPOCHS    = 30
# cfg.USE_WANDB = True

cfg.refresh()
cfg.summary()

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Verification de la plomberie (sans GPU ni donnees) ---
!python {REPO_DIR}/tests/test_pipeline.py

## Datasets COCO et table d'augmentation

In [ ]:
# --- Construction des datasets COCO des 5 folds (les deux arborescences) ---
# Optionnel ici : chaque notebook de modele le fait pour ses propres folds.
from aphids_det import cocoify
for fold in cfg.CV_FOLDS:
    for layout in ("flat", "coco"):
        ds, npos, nneg = cocoify.build_coco_fold(fold, layout=layout, verbose=False)
    print(f"fold {fold} : train {npos} pucerons / {nneg} fonds -> {ds.parent}")

In [ ]:
# --- Table de correspondance des augmentations (CSV a cote des resultats) ---
from aphids_det import augment
augment.write_csv()
augment.table()